# Appendix A2: The Daw Two-Step Task
## Model-Based vs. Model-Free Decision Making

**Spinning Up in Active Inference** | Appendix

---

In 2011, Nathaniel Daw and colleagues introduced a behavioral task that cleanly separates model-based from model-free decision-making in a single experiment. The **two-step task** has become the gold standard for measuring the balance between habitual and goal-directed control in humans, with implications for understanding addiction, OCD, and schizophrenia.

This notebook implements the task from scratch, builds three types of agents (model-free, model-based, and Active Inference), and shows how the classic 2×2 stay-probability analysis reveals fundamentally different decision strategies.

By the end of this notebook you will:
1. Understand the two-step task structure and why it dissociates MF from MB
2. Implement model-free, model-based, and hybrid agents
3. Reproduce the classic stay-probability analysis
4. See why Active Inference is inherently model-based
5. Connect the task to the Rosetta Stone (Module 13)

**Prerequisites:** Modules 4–5 (RL), Module 13 (Rosetta Stone).  
**Time:** ~75 minutes.

## A2.1 Why It Matters

Tolman showed that rats build **cognitive maps** — internal models of the world — rather than merely caching stimulus-response habits. But how do you *measure* this in a controlled experiment? You need a task where the two strategies make **opposite predictions** about behavior.

### Task Structure

The Daw two-step task has a simple but ingenious design:

1. **Stage 1:** Choose between two options (A or B)
2. **Probabilistic transition:** Each Stage 1 choice leads to one of two Stage 2 states with a **70/30** split:
   - Choice A → State 2A (70% *common*) or State 2B (30% *rare*)
   - Choice B → State 2B (70% *common*) or State 2A (30% *rare*)
3. **Stage 2:** Choose between two options within the reached state
4. **Reward:** Probabilistic, drawn from a **drifting Gaussian random walk** (so the best option changes over time)

### The Key Insight

What happens after a **rare** transition that leads to **reward**?

- **Model-free (habit):** **Stay** with the same Stage 1 choice. The action led to reward — reinforce it. The transition structure is invisible to cached values.
- **Model-based (planning):** **Switch** to the other Stage 1 choice. The rare transition means the *other* action *commonly* leads to the rewarding Stage 2 state.

### The 2×2 Stay-Probability Analysis

We cross **transition type** {Common, Rare} with **reward** {Rewarded, Unrewarded} and compute $P(\text{stay})$ — the probability of repeating the same Stage 1 choice on the next trial.

$$P(\text{stay}) = f(\text{transition type}, \text{reward})$$

| | Unrewarded | Rewarded |
|---|---|---|
| **Common** | low | high |
| **Rare** | ? | ? |

- **Model-free prediction:** Main effect of reward only. Both common and rare lines go up with reward (parallel lines). $P(\text{stay} \mid \text{rare, rewarded}) > P(\text{stay} \mid \text{rare, unrewarded})$
- **Model-based prediction:** Interaction between reward and transition type. After rare+rewarded, the agent *switches* (low stay). Lines cross. $P(\text{stay} \mid \text{rare, rewarded}) < P(\text{stay} \mid \text{rare, unrewarded})$

This is Tolman's cognitive map hypothesis, operationalized as a 2×2 interaction.

In [ ]:
# ── Setup ─────────────────────────────────────────────────────────────────────────
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch, FancyArrowPatch
import sys; sys.path.insert(0, '..')
from utils.plotting import COLORS, _setup_style, dual_perspective_box, plot_matrix_heatmap, plot_learning_curve
_setup_style()

np.random.seed(42)
print("Setup complete.")

## A2.2 Implementing the Task

We implement the Daw two-step task as a self-contained class. The key features:
- **Probabilistic transitions:** 70/30 common/rare split
- **Drifting rewards:** Four reward probabilities (2 Stage-2 states × 2 options) follow independent Gaussian random walks with reflecting boundaries at [0.25, 0.75]
- The drift ensures that the optimal strategy changes over time, forcing agents to continually re-learn

In [ ]:
# ── The Two-Step Task ────────────────────────────────────────────────────────────
class DawTwoStepTask:
    """Daw et al. (2011) two-step Markov decision task."""
    
    def __init__(self, p_common=0.7, drift_sd=0.025, seed=42):
        self.p_common = p_common
        self.drift_sd = drift_sd
        self.rng = np.random.RandomState(seed)
        # Four reward probabilities (2 Stage-2 states x 2 options each)
        self.reward_probs = self.rng.uniform(0.25, 0.75, size=4)
    
    def _drift_rewards(self):
        """Gaussian random walk with reflecting boundaries."""
        self.reward_probs += self.rng.normal(0, self.drift_sd, size=4)
        self.reward_probs = np.clip(self.reward_probs, 0.25, 0.75)
    
    def step(self, stage1_choice):
        """Run one trial given Stage 1 choice (0 or 1).
        
        Returns dict with stage1_choice, stage2_state, transition_type,
        stage2_choice (best), reward.
        """
        # Transition
        if self.rng.random() < self.p_common:
            stage2_state = stage1_choice  # common
            transition = 'common'
        else:
            stage2_state = 1 - stage1_choice  # rare
            transition = 'rare'
        
        # Stage 2: pick the better option in this state
        idx_base = stage2_state * 2
        stage2_choice = np.argmax(self.reward_probs[idx_base:idx_base + 2])
        
        # Reward
        reward = int(self.rng.random() < self.reward_probs[idx_base + stage2_choice])
        
        self._drift_rewards()
        
        return {
            'stage1_choice': stage1_choice,
            'stage2_state': stage2_state,
            'stage2_choice': stage2_choice,
            'transition': transition,
            'reward': reward,
        }

# Quick test
task = DawTwoStepTask()
for i in range(3):
    result = task.step(np.random.randint(2))
    print(f"Trial {i+1}: {result}")

In [ ]:
# ── Task Structure Diagram ───────────────────────────────────────────────────────
fig, ax = plt.subplots(1, 1, figsize=(10, 7))
ax.set_xlim(0, 10)
ax.set_ylim(0, 8)
ax.set_aspect('equal')
ax.axis('off')
ax.set_title('Daw Two-Step Task Structure', fontsize=15, fontweight='bold')

# Stage 1 box
s1_box = FancyBboxPatch((3.5, 6.2), 3, 1.2, boxstyle='round,pad=0.15',
                         facecolor=COLORS['neutral'], alpha=0.3, edgecolor='black', linewidth=2)
ax.add_patch(s1_box)
ax.text(5, 6.8, 'Stage 1\nChoice A or B', ha='center', va='center', fontsize=12, fontweight='bold')

# Stage 2 boxes
s2a_box = FancyBboxPatch((0.5, 3.2), 3, 1.2, boxstyle='round,pad=0.15',
                          facecolor=COLORS['rl'], alpha=0.25, edgecolor=COLORS['rl'], linewidth=2)
ax.add_patch(s2a_box)
ax.text(2, 3.8, 'Stage 2A\n2 options', ha='center', va='center', fontsize=11, fontweight='bold')

s2b_box = FancyBboxPatch((6.5, 3.2), 3, 1.2, boxstyle='round,pad=0.15',
                          facecolor=COLORS['aif'], alpha=0.25, edgecolor=COLORS['aif'], linewidth=2)
ax.add_patch(s2b_box)
ax.text(8, 3.8, 'Stage 2B\n2 options', ha='center', va='center', fontsize=11, fontweight='bold')

# Reward boxes
for x_pos, label in [(2, 'Reward\n(drifting)'), (8, 'Reward\n(drifting)')]:
    r_box = FancyBboxPatch((x_pos - 1.3, 0.8), 2.6, 1.0, boxstyle='round,pad=0.15',
                            facecolor=COLORS['reward'], alpha=0.25, edgecolor=COLORS['reward'], linewidth=2)
    ax.add_patch(r_box)
    ax.text(x_pos, 1.3, label, ha='center', va='center', fontsize=10, fontweight='bold')

# Arrows: common (solid, 70%) and rare (dashed, 30%)
# Choice A -> Stage 2A (common)
ax.annotate('', xy=(2.5, 4.4), xytext=(4.2, 6.2),
            arrowprops=dict(arrowstyle='->', color=COLORS['rl'], lw=2.5))
ax.text(2.8, 5.5, '70%', fontsize=10, color=COLORS['rl'], fontweight='bold')

# Choice A -> Stage 2B (rare)
ax.annotate('', xy=(7.0, 4.4), xytext=(4.5, 6.2),
            arrowprops=dict(arrowstyle='->', color=COLORS['rl'], lw=1.5, linestyle='dashed'))
ax.text(6.2, 5.7, '30%', fontsize=9, color=COLORS['rl'], fontstyle='italic')

# Choice B -> Stage 2B (common)
ax.annotate('', xy=(7.5, 4.4), xytext=(5.8, 6.2),
            arrowprops=dict(arrowstyle='->', color=COLORS['aif'], lw=2.5))
ax.text(7.0, 5.5, '70%', fontsize=10, color=COLORS['aif'], fontweight='bold')

# Choice B -> Stage 2A (rare)
ax.annotate('', xy=(3.0, 4.4), xytext=(5.5, 6.2),
            arrowprops=dict(arrowstyle='->', color=COLORS['aif'], lw=1.5, linestyle='dashed'))
ax.text(3.6, 5.7, '30%', fontsize=9, color=COLORS['aif'], fontstyle='italic')

# Stage 2 -> Reward
for x_pos in [2, 8]:
    ax.annotate('', xy=(x_pos, 1.8), xytext=(x_pos, 3.2),
                arrowprops=dict(arrowstyle='->', color=COLORS['reward'], lw=2))

# Legend
ax.plot([], [], '-', color='black', lw=2.5, label='Common (70%)')
ax.plot([], [], '--', color='black', lw=1.5, label='Rare (30%)')
ax.legend(loc='upper right', fontsize=11)

plt.tight_layout()
plt.show()

In [ ]:
# ── Reward Drift Visualization ──────────────────────────────────────────────────
task_viz = DawTwoStepTask(seed=42)
n_trials_viz = 200
reward_history = np.zeros((n_trials_viz, 4))

for t in range(n_trials_viz):
    reward_history[t] = task_viz.reward_probs.copy()
    task_viz._drift_rewards()

fig, ax = plt.subplots(figsize=(10, 4))
labels = ['State 2A, Option 1', 'State 2A, Option 2',
          'State 2B, Option 1', 'State 2B, Option 2']
colors_drift = [COLORS['rl'], COLORS['rl'], COLORS['aif'], COLORS['aif']]
styles = ['-', '--', '-', '--']

for i in range(4):
    ax.plot(reward_history[:, i], color=colors_drift[i], linestyle=styles[i],
            label=labels[i], linewidth=1.8, alpha=0.85)

ax.axhline(0.25, color='gray', linestyle=':', alpha=0.5, label='Boundaries')
ax.axhline(0.75, color='gray', linestyle=':', alpha=0.5)
ax.set_xlabel('Trial')
ax.set_ylabel('Reward Probability')
ax.set_title('Drifting Reward Probabilities (Gaussian Random Walk)', fontsize=13)
ax.legend(loc='upper right', fontsize=9)
ax.set_ylim(0.1, 0.9)
plt.tight_layout()
plt.show()

print("The optimal Stage 2 option changes over time, forcing continuous re-learning.")

## A2.3 Model-Free Agent

The model-free agent treats the two-step task as a flat stimulus-response mapping. It uses **SARSA-style TD learning** with an eligibility trace that propagates Stage 2 reward prediction errors back to Stage 1 values.

Key update equations (following Daw et al. 2011):

**Stage 2 update:**
$$\delta_2 = r - Q_2(s_2, a_2)$$
$$Q_2(s_2, a_2) \leftarrow Q_2(s_2, a_2) + \alpha_2 \cdot \delta_2$$

**Stage 1 update (with eligibility trace $\lambda$):**
$$\delta_1 = Q_2(s_2, a_2^*) - Q_1(a_1)$$
$$Q_1(a_1) \leftarrow Q_1(a_1) + \alpha_1 \cdot \delta_1 + \lambda \cdot \delta_2$$

The $\lambda$ parameter controls how much the Stage 2 RPE propagates directly to Stage 1 — this is the eligibility trace that makes MF agents credit Stage 1 actions for Stage 2 rewards, **regardless of the transition**.

In [ ]:
# ── Model-Free Agent ─────────────────────────────────────────────────────────────
class ModelFreeAgent:
    """Model-free agent with eligibility trace (Daw et al. 2011)."""
    def __init__(self, alpha1=0.5, alpha2=0.5, lam=0.5, beta1=5.0, beta2=5.0, p=0.0):
        self.alpha1 = alpha1
        self.alpha2 = alpha2
        self.lam = lam
        self.beta1 = beta1  # Stage 1 inverse temperature
        self.beta2 = beta2  # Stage 2 inverse temperature
        self.p = p          # perseveration
        self.Q1 = np.zeros(2)
        self.Q2 = np.zeros((2, 2))  # [state, action]
        self.last_choice = None
    
    def choose_stage1(self):
        logits = self.beta1 * self.Q1.copy()
        if self.last_choice is not None:
            logits[self.last_choice] += self.p
        probs = np.exp(logits - np.max(logits))
        probs /= probs.sum()
        return np.random.choice(2, p=probs)
    
    def choose_stage2(self, state):
        logits = self.beta2 * self.Q2[state]
        probs = np.exp(logits - np.max(logits))
        probs /= probs.sum()
        return np.random.choice(2, p=probs)
    
    def update(self, s1_choice, s2_state, s2_choice, reward):
        # Stage 2 RPE
        rpe2 = reward - self.Q2[s2_state, s2_choice]
        self.Q2[s2_state, s2_choice] += self.alpha2 * rpe2
        
        # Stage 1 update with eligibility trace
        rpe1 = self.Q2[s2_state].max() - self.Q1[s1_choice]
        self.Q1[s1_choice] += self.alpha1 * rpe1 + self.lam * rpe2
        
        self.last_choice = s1_choice

print("ModelFreeAgent defined.")

In [ ]:
# ── Experiment Helpers ───────────────────────────────────────────────────────────
def run_experiment(agent, task, n_trials=1000):
    """Run agent on task, return trial history."""
    history = []
    for t in range(n_trials):
        s1 = agent.choose_stage1()
        result = task.step(s1)
        s2_choice = agent.choose_stage2(result['stage2_state'])
        # Re-determine reward using the agent's actual Stage 2 choice
        idx_base = result['stage2_state'] * 2
        reward = int(task.rng.random() < task.reward_probs[idx_base + s2_choice])
        agent.update(s1, result['stage2_state'], s2_choice, reward)
        history.append({
            'trial': t,
            'stage1_choice': s1,
            'stage2_state': result['stage2_state'],
            'transition': result['transition'],
            'reward': reward,
            'stayed': None  # filled in post-hoc
        })
    
    # Compute stay/switch
    for i in range(1, len(history)):
        history[i]['stayed'] = int(history[i]['stage1_choice'] == history[i-1]['stage1_choice'])
    
    return history


def compute_stay_probabilities(history):
    """Classic 2x2 analysis: P(stay) for {common,rare} x {rewarded,unrewarded}."""
    conditions = {
        ('common', 1): [], ('common', 0): [],
        ('rare', 1): [], ('rare', 0): [],
    }
    for i in range(1, len(history)):
        key = (history[i-1]['transition'], history[i-1]['reward'])
        if history[i]['stayed'] is not None:
            conditions[key].append(history[i]['stayed'])
    
    result = {}
    for key, vals in conditions.items():
        result[key] = np.mean(vals) if vals else 0.5
    return result


def plot_stay_probabilities(stay_probs, title="Stay Probability", ax=None):
    """Plot the classic 2x2 stay-probability figure."""
    if ax is None:
        fig, ax = plt.subplots(figsize=(6, 4))
    
    x = [0, 1]
    common = [stay_probs[('common', 0)], stay_probs[('common', 1)]]
    rare = [stay_probs[('rare', 0)], stay_probs[('rare', 1)]]
    
    ax.plot(x, common, 'o-', color=COLORS['rl'], linewidth=2.5, markersize=10, label='Common')
    ax.plot(x, rare, 's--', color=COLORS['aif'], linewidth=2.5, markersize=10, label='Rare')
    
    ax.set_xticks(x)
    ax.set_xticklabels(['Unrewarded', 'Rewarded'])
    ax.set_ylabel('P(stay)')
    ax.set_ylim(0.3, 0.9)
    ax.set_title(title)
    ax.legend()
    return ax


def interaction_effect(stay_probs):
    """Compute the interaction: how much does transition type modulate the reward effect?
    
    Positive = model-based (rare+rewarded -> switch).
    Near zero = model-free (parallel lines).
    """
    common_effect = stay_probs[('common', 1)] - stay_probs[('common', 0)]
    rare_effect = stay_probs[('rare', 1)] - stay_probs[('rare', 0)]
    return common_effect - rare_effect

print("Helper functions defined.")

In [ ]:
# ── Train Model-Free Agent ───────────────────────────────────────────────────────
np.random.seed(42)
mf_agent = ModelFreeAgent(alpha1=0.4, alpha2=0.4, lam=0.6, beta1=5.0, beta2=5.0, p=0.2)
mf_task = DawTwoStepTask(seed=123)
mf_history = run_experiment(mf_agent, mf_task, n_trials=1000)

mf_stay = compute_stay_probabilities(mf_history)

fig, ax = plt.subplots(figsize=(6, 4))
plot_stay_probabilities(mf_stay, title='Model-Free Agent: Stay Probabilities', ax=ax)
ax.text(0.5, 0.85, f'Interaction = {interaction_effect(mf_stay):.3f}\n(near zero = MF)',
        ha='center', fontsize=10, color=COLORS['neutral'],
        transform=ax.transAxes, bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))
plt.tight_layout()
plt.show()

print(f"\nStay probabilities:")
for key, val in sorted(mf_stay.items()):
    print(f"  {key[0]:>6s}, {'rewarded' if key[1] else 'unrewarded':>12s}: P(stay) = {val:.3f}")
print(f"\nInteraction effect: {interaction_effect(mf_stay):.3f}")
print("Near-parallel lines -> main effect of reward only (model-free signature).")

## A2.4 Model-Based Agent

The model-based agent maintains an explicit **transition model** $T(s_2 | a_1)$ and uses **backward induction** to compute Stage 1 values:

$$Q_1^{\text{MB}}(a_1) = \sum_{s_2} T(s_2 | a_1) \cdot \max_{a_2} Q_2(s_2, a_2)$$

The transition model is learned from observed transition counts. Because the agent knows *which* Stage 2 state each Stage 1 action tends to lead to, it can reason about the consequence of rare transitions:

- After a **rare** transition to State 2A that yields reward, the MB agent knows that the **other** Stage 1 action *commonly* leads to State 2A
- Therefore it **switches** — the opposite of the MF agent

In [ ]:
# ── Model-Based Agent ────────────────────────────────────────────────────────────
class ModelBasedAgent:
    """Model-based agent that learns transition model and uses backward induction."""
    def __init__(self, alpha2=0.5, beta1=5.0, beta2=5.0, p=0.0):
        self.alpha2 = alpha2
        self.beta1 = beta1
        self.beta2 = beta2
        self.p = p
        self.Q2 = np.zeros((2, 2))  # [state, action]
        # Transition counts: T_counts[a1, s2] = number of times action a1 led to state s2
        self.T_counts = np.ones((2, 2))  # Laplace prior (start with 1)
        self.last_choice = None
    
    def _get_transition_probs(self):
        """Normalize transition counts to probabilities."""
        T = self.T_counts.copy()
        T = T / T.sum(axis=1, keepdims=True)
        return T
    
    def _compute_Q1_MB(self):
        """Backward induction: Q1(a) = sum_s T(s|a) * max_a' Q2(s, a')."""
        T = self._get_transition_probs()
        Q2_max = np.max(self.Q2, axis=1)  # [2] best Q at each Stage 2 state
        return T @ Q2_max  # [2] Q1 for each Stage 1 action
    
    def choose_stage1(self):
        Q1 = self._compute_Q1_MB()
        logits = self.beta1 * Q1
        if self.last_choice is not None:
            logits[self.last_choice] += self.p
        probs = np.exp(logits - np.max(logits))
        probs /= probs.sum()
        return np.random.choice(2, p=probs)
    
    def choose_stage2(self, state):
        logits = self.beta2 * self.Q2[state]
        probs = np.exp(logits - np.max(logits))
        probs /= probs.sum()
        return np.random.choice(2, p=probs)
    
    def update(self, s1_choice, s2_state, s2_choice, reward):
        # Update transition model
        self.T_counts[s1_choice, s2_state] += 1
        
        # Stage 2 TD update
        rpe2 = reward - self.Q2[s2_state, s2_choice]
        self.Q2[s2_state, s2_choice] += self.alpha2 * rpe2
        
        self.last_choice = s1_choice

print("ModelBasedAgent defined.")

In [ ]:
# ── Train Model-Based Agent ──────────────────────────────────────────────────────
np.random.seed(42)
mb_agent = ModelBasedAgent(alpha2=0.4, beta1=5.0, beta2=5.0, p=0.2)
mb_task = DawTwoStepTask(seed=123)
mb_history = run_experiment(mb_agent, mb_task, n_trials=1000)

mb_stay = compute_stay_probabilities(mb_history)

fig, ax = plt.subplots(figsize=(6, 4))
plot_stay_probabilities(mb_stay, title='Model-Based Agent: Stay Probabilities', ax=ax)
ax.text(0.5, 0.85, f'Interaction = {interaction_effect(mb_stay):.3f}\n(large positive = MB)',
        ha='center', fontsize=10, color=COLORS['neutral'],
        transform=ax.transAxes, bbox=dict(boxstyle='round', facecolor='lightblue', alpha=0.5))
plt.tight_layout()
plt.show()

print(f"\nStay probabilities:")
for key, val in sorted(mb_stay.items()):
    print(f"  {key[0]:>6s}, {'rewarded' if key[1] else 'unrewarded':>12s}: P(stay) = {val:.3f}")
print(f"\nInteraction effect: {interaction_effect(mb_stay):.3f}")
print("Crossing lines -> interaction between reward and transition (model-based signature).")

In [ ]:
# ── Side-by-Side Comparison ──────────────────────────────────────────────────────
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4.5))

plot_stay_probabilities(mf_stay, title='Model-Free Agent', ax=ax1)
ax1.text(0.5, 0.38, 'Parallel lines\n(main effect of reward)',
         ha='center', fontsize=9, color=COLORS['danger'], transform=ax1.transAxes)

plot_stay_probabilities(mb_stay, title='Model-Based Agent', ax=ax2)
ax2.text(0.5, 0.38, 'Crossing lines\n(reward x transition interaction)',
         ha='center', fontsize=9, color=COLORS['reward'], transform=ax2.transAxes)

fig.suptitle('The Classic Daw et al. (2011) Dissociation', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print(f"Interaction effects:  MF = {interaction_effect(mf_stay):.3f}   MB = {interaction_effect(mb_stay):.3f}")

## A2.5 Hybrid Agent

Human data typically shows *both* model-free and model-based signatures. Daw et al. (2011) proposed a **hybrid** agent that blends the two systems:

$$Q_1^{\text{hybrid}}(a) = w \cdot Q_1^{\text{MB}}(a) + (1 - w) \cdot Q_1^{\text{MF}}(a)$$

where $w \in [0, 1]$ is the **model-based weight**.

This mixing parameter has become a key individual-difference measure in computational psychiatry:

| Population | Typical $w$ | Interpretation |
|---|---|---|
| Healthy adults | 0.4–0.7 | Mixed strategy |
| Under cognitive load | Lower $w$ | MB requires working memory |
| Compulsive disorders (OCD) | Lower $w$ | Excessive habitual control |
| Alcohol dependence | Lower $w$ | Shift toward MF |
| After dopamine depletion | Lower $w$ | MB depends on dopamine |

In [ ]:
# ── Hybrid Agent ─────────────────────────────────────────────────────────────────
class HybridAgent:
    """Hybrid MB/MF agent (Daw et al. 2011).
    
    w=0 is pure model-free, w=1 is pure model-based.
    """
    def __init__(self, w=0.5, alpha1=0.5, alpha2=0.5, lam=0.5,
                 beta1=5.0, beta2=5.0, p=0.0):
        self.w = w
        self.alpha1 = alpha1
        self.alpha2 = alpha2
        self.lam = lam
        self.beta1 = beta1
        self.beta2 = beta2
        self.p = p
        # MF values
        self.Q1_MF = np.zeros(2)
        self.Q2 = np.zeros((2, 2))
        # MB transition model
        self.T_counts = np.ones((2, 2))  # Laplace prior
        self.last_choice = None
    
    def _get_transition_probs(self):
        T = self.T_counts.copy()
        return T / T.sum(axis=1, keepdims=True)
    
    def _compute_Q1_MB(self):
        T = self._get_transition_probs()
        Q2_max = np.max(self.Q2, axis=1)
        return T @ Q2_max
    
    def choose_stage1(self):
        Q1_MB = self._compute_Q1_MB()
        Q1_hybrid = self.w * Q1_MB + (1 - self.w) * self.Q1_MF
        logits = self.beta1 * Q1_hybrid
        if self.last_choice is not None:
            logits[self.last_choice] += self.p
        probs = np.exp(logits - np.max(logits))
        probs /= probs.sum()
        return np.random.choice(2, p=probs)
    
    def choose_stage2(self, state):
        logits = self.beta2 * self.Q2[state]
        probs = np.exp(logits - np.max(logits))
        probs /= probs.sum()
        return np.random.choice(2, p=probs)
    
    def update(self, s1_choice, s2_state, s2_choice, reward):
        # Update transition model
        self.T_counts[s1_choice, s2_state] += 1
        
        # Stage 2 TD update
        rpe2 = reward - self.Q2[s2_state, s2_choice]
        self.Q2[s2_state, s2_choice] += self.alpha2 * rpe2
        
        # Stage 1 MF update with eligibility trace
        rpe1 = self.Q2[s2_state].max() - self.Q1_MF[s1_choice]
        self.Q1_MF[s1_choice] += self.alpha1 * rpe1 + self.lam * rpe2
        
        self.last_choice = s1_choice

print("HybridAgent defined.")

In [ ]:
# ── Sweep w Across [0, 0.25, 0.5, 0.75, 1.0] ─────────────────────────────────
w_values = [0.0, 0.25, 0.5, 0.75, 1.0]
fig, axes = plt.subplots(1, 5, figsize=(20, 4), sharey=True)

for idx, w in enumerate(w_values):
    np.random.seed(42)
    agent = HybridAgent(w=w, alpha1=0.4, alpha2=0.4, lam=0.6, beta1=5.0, beta2=5.0, p=0.2)
    task = DawTwoStepTask(seed=123)
    history = run_experiment(agent, task, n_trials=1000)
    stay = compute_stay_probabilities(history)
    
    plot_stay_probabilities(stay, title=f'w = {w:.2f}', ax=axes[idx])
    ie = interaction_effect(stay)
    axes[idx].text(0.5, 0.12, f'IE = {ie:.3f}', ha='center', fontsize=9,
                   transform=axes[idx].transAxes,
                   bbox=dict(boxstyle='round', facecolor='lightyellow', alpha=0.7))
    if idx > 0:
        axes[idx].set_ylabel('')

fig.suptitle('Hybrid Agent: Model-Based Weight Sweep', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print("w=0 (pure MF): parallel lines. w=1 (pure MB): crossing lines. Humans are in between.")

In [ ]:
# ── Fine-Grained Interaction Effect vs. w ─────────────────────────────────────
w_sweep = np.linspace(0, 1, 21)
ie_sweep = []

for w in w_sweep:
    np.random.seed(42)
    agent = HybridAgent(w=w, alpha1=0.4, alpha2=0.4, lam=0.6, beta1=5.0, beta2=5.0, p=0.2)
    task = DawTwoStepTask(seed=123)
    history = run_experiment(agent, task, n_trials=1000)
    stay = compute_stay_probabilities(history)
    ie_sweep.append(interaction_effect(stay))

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(w_sweep, ie_sweep, 'o-', color=COLORS['epistemic'], linewidth=2.5, markersize=7)
ax.axhline(0, color='gray', linestyle=':', alpha=0.5)

# Annotate typical human range
ax.axvspan(0.4, 0.7, alpha=0.15, color=COLORS['highlight'], label='Typical human range')
ax.annotate('Typical\nhuman\nrange', xy=(0.55, ie_sweep[11]), fontsize=10,
            ha='center', color=COLORS['neutral'])

ax.set_xlabel('Model-Based Weight (w)', fontsize=12)
ax.set_ylabel('Interaction Effect', fontsize=12)
ax.set_title('Interaction Effect vs. Model-Based Weight', fontsize=13)
ax.legend(loc='upper left')
plt.tight_layout()
plt.show()

print(f"IE at w=0 (pure MF): {ie_sweep[0]:.3f}")
print(f"IE at w=1 (pure MB): {ie_sweep[-1]:.3f}")
print(f"IE at w=0.5 (typical human): {ie_sweep[10]:.3f}")

In [ ]:
# ── Hybrid Agent: Summary Statistics ─────────────────────────────────────────
# Run the hybrid agent at w=0.5 (typical human) and print a full summary.

np.random.seed(42)
hybrid_agent = HybridAgent(w=0.5, alpha1=0.4, alpha2=0.4, lam=0.6, beta1=5.0, beta2=5.0, p=0.2)
hybrid_task = DawTwoStepTask(seed=123)
hybrid_history = run_experiment(hybrid_agent, hybrid_task, n_trials=1000)
hybrid_stay = compute_stay_probabilities(hybrid_history)

print("Hybrid Agent (w=0.5) — Typical Human Parameterization")
print("=" * 55)
print(f"\nStay probabilities:")
for key, val in sorted(hybrid_stay.items()):
    print(f"  {key[0]:>6s}, {'rewarded' if key[1] else 'unrewarded':>12s}: P(stay) = {val:.3f}")
print(f"\nInteraction effect: {interaction_effect(hybrid_stay):.3f}")
print(f"Mean reward rate: {np.mean([h['reward'] for h in hybrid_history]):.3f}")
print(f"\nThe hybrid agent shows BOTH signatures:")
print(f"  - Main effect of reward (MF component)")
print(f"  - Partial interaction (MB component)")
print(f"This matches typical human behavioral data.")

## A2.6 Active Inference Perspective

Active Inference is **inherently model-based**. There is no AIF analog of a model-free agent, because the core computation — expected free energy — *requires* a transition model (the B-matrix).

### Why AIF Must Be Model-Based

Expected free energy (EFE) for a policy $\pi$ at timestep $\tau$:

$$G(\pi) = \sum_{\tau} G(\pi, \tau)$$

where each term decomposes as:

$$G(\pi, \tau) = \underbrace{-\sum_{s'} Q(s' | \pi) \ln P(o' | s') \cdot C(o')}_{\text{pragmatic: seek preferred outcomes}} + \underbrace{\mathbb{E}_{Q}[H[P(o'|s')]]}_{\text{epistemic: reduce uncertainty}}$$

The critical piece: computing $Q(s' | \pi)$ requires the **B-matrix** (transition model):

$$Q(s' | \pi) = \sum_s B(s' | s, a_\pi) \cdot Q(s)$$

Without a transition model, EFE cannot be computed. **There is no model-free Active Inference.**

### AIF on the Two-Step Task

For the two-step task, the AIF agent has:
- **B-matrix:** Encodes the 70/30 transition structure
- **C-vector:** Preference for reward (high C for rewarded outcomes)
- **Policy evaluation:** EFE naturally accounts for transition probabilities

The result: AIF shows the model-based signature (crossing lines) because it *must* use the transition model to evaluate actions.

In [ ]:
# ── Active Inference Agent ───────────────────────────────────────────────────────
class ActiveInferenceAgent:
    """Active Inference agent for the two-step task.
    
    Uses B-matrix (transition model) and C-vector (preferences)
    to compute expected free energy for Stage 1 policy selection.
    """
    def __init__(self, alpha_B=0.1, alpha_C=0.3, beta1=5.0, beta2=5.0, p=0.0):
        self.alpha_B = alpha_B  # B-matrix learning rate
        self.alpha_C = alpha_C  # C-vector (reward model) learning rate
        self.beta1 = beta1
        self.beta2 = beta2
        self.p = p
        
        # B-matrix: transition model B[a1] is P(s2 | a1)
        # Initialize with weak prior (uniform + small common bias)
        self.B_counts = np.ones((2, 2)) * 2  # B_counts[a1, s2]
        
        # Learned reward expectations per (state, action): proxy for C-vector alignment
        self.Q2 = np.zeros((2, 2))  # Q2[s2, a2]
        
        self.last_choice = None
    
    def _get_B(self):
        """Normalize B-matrix counts to transition probabilities."""
        B = self.B_counts.copy()
        return B / B.sum(axis=1, keepdims=True)
    
    def _compute_EFE(self):
        """Compute (negative) expected free energy for each Stage 1 action.
        
        G(a1) = -sum_s2 B(s2|a1) * max_a2 Q2(s2, a2)
        
        We return -G so that higher values = better actions (for softmax).
        """
        B = self._get_B()
        Q2_max = np.max(self.Q2, axis=1)  # best expected reward per state
        # Pragmatic value: expected reward under transition model
        neg_G = B @ Q2_max
        return neg_G
    
    def choose_stage1(self):
        neg_G = self._compute_EFE()
        logits = self.beta1 * neg_G
        if self.last_choice is not None:
            logits[self.last_choice] += self.p
        probs = np.exp(logits - np.max(logits))
        probs /= probs.sum()
        return np.random.choice(2, p=probs)
    
    def choose_stage2(self, state):
        logits = self.beta2 * self.Q2[state]
        probs = np.exp(logits - np.max(logits))
        probs /= probs.sum()
        return np.random.choice(2, p=probs)
    
    def update(self, s1_choice, s2_state, s2_choice, reward):
        # Update B-matrix (transition model)
        self.B_counts[s1_choice, s2_state] += 1
        
        # Update reward expectations (proxy for C-vector alignment)
        rpe = reward - self.Q2[s2_state, s2_choice]
        self.Q2[s2_state, s2_choice] += self.alpha_C * rpe
        
        self.last_choice = s1_choice

print("ActiveInferenceAgent defined.")

In [ ]:
# ── Train AIF Agent ──────────────────────────────────────────────────────────────
np.random.seed(42)
aif_agent = ActiveInferenceAgent(alpha_B=0.1, alpha_C=0.4, beta1=5.0, beta2=5.0, p=0.2)
aif_task = DawTwoStepTask(seed=123)
aif_history = run_experiment(aif_agent, aif_task, n_trials=1000)

aif_stay = compute_stay_probabilities(aif_history)

fig, ax = plt.subplots(figsize=(6, 4))
plot_stay_probabilities(aif_stay, title='Active Inference Agent: Stay Probabilities', ax=ax)
ax.text(0.5, 0.85, f'Interaction = {interaction_effect(aif_stay):.3f}\n(positive = model-based)',
        ha='center', fontsize=10, color=COLORS['neutral'],
        transform=ax.transAxes, bbox=dict(boxstyle='round', facecolor='#FFE0D0', alpha=0.5))
plt.tight_layout()
plt.show()

print(f"\nAIF stay probabilities:")
for key, val in sorted(aif_stay.items()):
    print(f"  {key[0]:>6s}, {'rewarded' if key[1] else 'unrewarded':>12s}: P(stay) = {val:.3f}")
print(f"\nInteraction effect: {interaction_effect(aif_stay):.3f}")
print("AIF shows model-based signature because EFE *requires* the B-matrix.")

In [ ]:
# ── Three-Way Comparison ─────────────────────────────────────────────────────────
fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(16, 4.5), sharey=True)

plot_stay_probabilities(mf_stay, title='Model-Free', ax=ax1)
plot_stay_probabilities(mb_stay, title='Model-Based', ax=ax2)
plot_stay_probabilities(aif_stay, title='Active Inference', ax=ax3)

for ax, label in zip([ax1, ax2, ax3], ['MF', 'MB', 'AIF']):
    ax.set_ylabel('')

ax1.set_ylabel('P(stay)')

fig.suptitle('Three Decision Strategies on the Two-Step Task', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# ── Interaction Effect Bar Chart ──────────────────────────────────────────────────
ie_values = [
    interaction_effect(mf_stay),
    interaction_effect(mb_stay),
    interaction_effect(aif_stay)
]
labels = ['Model-Free', 'Model-Based', 'Active Inference']
bar_colors = [COLORS['rl'], COLORS['reward'], COLORS['aif']]

fig, ax = plt.subplots(figsize=(7, 4))
bars = ax.bar(labels, ie_values, color=bar_colors, alpha=0.85, edgecolor='black', linewidth=1.2)
ax.axhline(0, color='gray', linestyle=':', alpha=0.5)
ax.set_ylabel('Interaction Effect', fontsize=12)
ax.set_title('Model-Based Index: Interaction Effect Comparison', fontsize=13)

for bar, val in zip(bars, ie_values):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.01,
            f'{val:.3f}', ha='center', fontsize=11, fontweight='bold')

ax.text(0.5, 0.92, 'Higher = more model-based', ha='center', fontsize=10,
        color=COLORS['neutral'], transform=ax.transAxes)
plt.tight_layout()
plt.show()

In [ ]:
# ── Dual Perspective Box ─────────────────────────────────────────────────────────
dual_perspective_box(
    rl_text=(
        "The model-free agent caches Q-values from reward experience. "
        "The eligibility trace propagates reward credit to Stage 1 actions "
        "<b>regardless of how the agent got to Stage 2</b>. Transition structure "
        "is invisible to cached values. This is Thorndike's Law of Effect: "
        "the action was followed by reward, so repeat it."
    ),
    aif_text=(
        "The Active Inference agent <b>must</b> use a transition model (B-matrix) "
        "to compute expected free energy. EFE = sum over future states weighted by "
        "B(s'|s,a). Without B, there is no EFE and no policy selection. "
        "This is why AIF is inherently model-based: the B-matrix <i>is</i> Tolman's "
        "cognitive map, and policy selection <i>is</i> mental simulation."
    ),
    title="Model-Free Caching vs. Active Inference Planning"
)

In [ ]:
# ── B-Matrix and C-Vector Visualization ────────────────────────────────────────
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4.5))

# B-matrix: learned transition model
B_learned = aif_agent._get_B()
plot_matrix_heatmap(
    B_learned,
    title='Learned B-Matrix (Transition Model)\nP(Stage 2 state | Stage 1 action)',
    row_labels=['Action A', 'Action B'],
    col_labels=['State 2A', 'State 2B'],
    cmap='Blues',
    ax=ax1
)

# C-vector proxy: learned reward expectations
Q2_flat = aif_agent.Q2.flatten()
c_labels = ['2A-Opt1', '2A-Opt2', '2B-Opt1', '2B-Opt2']
bar_colors_c = [COLORS['rl'], COLORS['rl'], COLORS['aif'], COLORS['aif']]
ax2.bar(c_labels, Q2_flat, color=bar_colors_c, alpha=0.85, edgecolor='black')
ax2.set_ylabel('Expected Reward (Q2)', fontsize=11)
ax2.set_title('Learned Reward Expectations\n(proxy for C-vector alignment)', fontsize=13)
ax2.set_ylim(0, 1)

for i, v in enumerate(Q2_flat):
    ax2.text(i, v + 0.02, f'{v:.2f}', ha='center', fontsize=10)

plt.tight_layout()
plt.show()

print(f"Learned transition model:")
print(f"  Action A -> State 2A: {B_learned[0,0]:.3f} (true: 0.70)")
print(f"  Action A -> State 2B: {B_learned[0,1]:.3f} (true: 0.30)")
print(f"  Action B -> State 2A: {B_learned[1,0]:.3f} (true: 0.30)")
print(f"  Action B -> State 2B: {B_learned[1,1]:.3f} (true: 0.70)")

## A2.7 NeuroGym Connection

The [NeuroGym](https://github.com/neurogym/neurogym) library provides a Gymnasium-compatible implementation of the Daw two-step task as `DawTwoStep-v0`. This enables:

- Training **RNNs** on the task and analyzing whether they learn model-based or model-free strategies
- Comparing **architectural effects**: Do LSTMs, GRUs, and transformers differ in their model-based index?
- **Individual differences**: Can network hyperparameters predict the MF/MB balance, analogous to psychiatric phenotypes?

See Ji-An, Benna & Mattar (2025, *Nature*) for evidence that tiny RNNs (1-4 units) trained on reward-learning tasks can discover novel behavioral strategies invisible to classical model fitting.

In [ ]:
# ── NeuroGym Connection ──────────────────────────────────────────────────────────
try:
    import neurogym as ngym
    env = ngym.make('DawTwoStep-v0')
    obs, info = env.reset()
    print(f"NeuroGym DawTwoStep-v0 loaded!")
    print(f"  Observation shape: {obs.shape}")
    print(f"  Action space: {env.action_space}")
    print(f"\nThis Gymnasium-compatible wrapper can be used with any")
    print(f"deep RL framework (PyTorch, JAX) to train RNNs on the task.")
    print(f"\nKey analysis: compute the stay-probability 2x2 from the")
    print(f"trained RNN's behavior to classify it as MF, MB, or hybrid.")
except ImportError:
    print("NeuroGym not installed. Install with: pip install neurogym")
    print("\nThe DawTwoStep-v0 environment provides a Gymnasium-compatible")
    print("wrapper around the same task we built from scratch above.")
    print("It enables training RNNs and analyzing whether they develop")
    print("model-based or model-free strategies.")
    print("\nSee Appendix A1 for more NeuroGym cognitive tasks.")

## A2.8 Exercises

### Exercise 1 (Guided): Effect of Reward Drift Rate

How does the drift rate affect the MF vs. MB advantage? Faster drift means the environment changes more quickly, punishing agents that rely on stale cached values.

Compare three drift rates: `drift_sd = [0.01, 0.025, 0.05]`. For each, run both MF and MB agents for 1000 trials and plot their cumulative reward curves.

In [ ]:
# ── Exercise 1: Drift Rate Comparison ─────────────────────────────────────────
drift_rates = [0.01, 0.025, 0.05]
n_trials_ex = 1000

fig, axes = plt.subplots(1, 3, figsize=(16, 4.5), sharey=True)

for idx, drift_sd in enumerate(drift_rates):
    # Model-free
    np.random.seed(42)
    mf_ag = ModelFreeAgent(alpha1=0.4, alpha2=0.4, lam=0.6, beta1=5.0, beta2=5.0)
    mf_tk = DawTwoStepTask(seed=123, drift_sd=drift_sd)
    mf_h = run_experiment(mf_ag, mf_tk, n_trials=n_trials_ex)
    mf_rewards = [h['reward'] for h in mf_h]
    
    # Model-based
    np.random.seed(42)
    mb_ag = ModelBasedAgent(alpha2=0.4, beta1=5.0, beta2=5.0)
    mb_tk = DawTwoStepTask(seed=123, drift_sd=drift_sd)
    mb_h = run_experiment(mb_ag, mb_tk, n_trials=n_trials_ex)
    mb_rewards = [h['reward'] for h in mb_h]
    
    # Smoothed reward curves
    window = 50
    mf_smooth = np.convolve(mf_rewards, np.ones(window)/window, mode='valid')
    mb_smooth = np.convolve(mb_rewards, np.ones(window)/window, mode='valid')
    
    axes[idx].plot(mf_smooth, color=COLORS['rl'], linewidth=2, label='Model-Free')
    axes[idx].plot(mb_smooth, color=COLORS['reward'], linewidth=2, label='Model-Based')
    axes[idx].set_title(f'Drift SD = {drift_sd}', fontsize=12)
    axes[idx].set_xlabel('Trial')
    axes[idx].legend(fontsize=9)
    axes[idx].set_ylim(0.3, 0.85)
    
    mf_mean = np.mean(mf_rewards)
    mb_mean = np.mean(mb_rewards)
    axes[idx].text(0.5, 0.12, f'MF: {mf_mean:.3f}  MB: {mb_mean:.3f}',
                   ha='center', fontsize=9, transform=axes[idx].transAxes,
                   bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

axes[0].set_ylabel('Reward Rate (smoothed)', fontsize=11)
fig.suptitle('Effect of Reward Drift Rate on Agent Performance', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print("Faster drift penalizes model-free agents more: stale cached values become wrong faster.")
print("Model-based agents re-plan each trial using the transition model, so drift rate matters less.")

### Exercise 2 (Intermediate): Three-Stage Task

Extend the two-step task to a **three-stage** task with an additional level of probabilistic transitions. This makes the MB/MF dissociation even stronger, because the model-free agent has to propagate credit through two stochastic transitions.

Scaffold provided below. Fill in the `step()` method.

In [ ]:
# ── Exercise 2: Three-Stage Task Scaffold ─────────────────────────────────────
class ThreeStageTask:
    """Extended three-stage Markov decision task.
    
    Stage 1 -> (70/30) -> Stage 2 -> (70/30) -> Stage 3 -> Reward
    """
    def __init__(self, p_common=0.7, drift_sd=0.025, seed=42):
        self.p_common = p_common
        self.drift_sd = drift_sd
        self.rng = np.random.RandomState(seed)
        # 4 Stage-3 states x 2 options = 8 reward probabilities
        self.reward_probs = self.rng.uniform(0.25, 0.75, size=8)
    
    def _drift_rewards(self):
        self.reward_probs += self.rng.normal(0, self.drift_sd, size=8)
        self.reward_probs = np.clip(self.reward_probs, 0.25, 0.75)
    
    def step(self, stage1_choice):
        """TODO: Implement three-stage transition and reward logic.
        
        Hints:
        - Stage 1 choice (0 or 1) -> Stage 2 state (0 or 1) via 70/30
        - Stage 2 choice (0 or 1) combined with Stage 2 state -> Stage 3 state (0..3) via 70/30
        - Stage 3: pick best of 2 options, get probabilistic reward
        
        Return a dict with all intermediate states and transitions.
        """
        # YOUR CODE HERE
        # Stage 1 -> Stage 2 transition
        if self.rng.random() < self.p_common:
            stage2_state = stage1_choice
            trans1 = 'common'
        else:
            stage2_state = 1 - stage1_choice
            trans1 = 'rare'
        
        # Stage 2 choice -> Stage 3 transition
        stage2_choice = 0  # placeholder: replace with agent's choice
        stage3_base = stage2_state * 2
        if self.rng.random() < self.p_common:
            stage3_state = stage3_base + stage2_choice
            trans2 = 'common'
        else:
            stage3_state = stage3_base + (1 - stage2_choice)
            trans2 = 'rare'
        
        # Stage 3 reward
        idx_base = stage3_state * 2
        stage3_choice = np.argmax(self.reward_probs[idx_base:idx_base + 2])
        reward = int(self.rng.random() < self.reward_probs[idx_base + stage3_choice])
        
        self._drift_rewards()
        
        return {
            'stage1_choice': stage1_choice,
            'stage2_state': stage2_state, 'transition1': trans1,
            'stage2_choice': stage2_choice,
            'stage3_state': stage3_state, 'transition2': trans2,
            'stage3_choice': stage3_choice,
            'reward': reward,
        }

# Test
task3 = ThreeStageTask()
result3 = task3.step(0)
print(f"Three-stage test: {result3}")
print("\nChallenge: build MF and MB agents for this task and compare stay probabilities.")

### Exercise 3 (Open-Ended): Play the Task Yourself

The code below implements a simple text-based version of the two-step task. Uncomment and run it in an interactive session to experience the task firsthand.

**Questions to reflect on:**
- After a *rare* transition that leads to reward, do you stay or switch?
- Do you find yourself building a mental model of the transition structure?
- How many trials does it take before you notice the 70/30 split?

In [ ]:
# ── Exercise 3: Play the Task Yourself (Interactive) ─────────────────────────
# Uncomment the code below and run in an interactive Jupyter session.
# It will prompt you for Stage 1 choices and show outcomes.

# def play_two_step(n_trials=20):
#     """Play the two-step task interactively."""
#     task = DawTwoStepTask(seed=None)  # random seed for each game
#     history = []
#     total_reward = 0
#     
#     print("=" * 50)
#     print("THE DAW TWO-STEP TASK")
#     print("=" * 50)
#     print(f"You have {n_trials} trials. Maximize your reward!")
#     print("Choice A commonly (70%) leads to State 2A")
#     print("Choice B commonly (70%) leads to State 2B")
#     print()
#     
#     for t in range(n_trials):
#         choice = input(f"Trial {t+1}: Choose A or B: ").strip().upper()
#         if choice not in ['A', 'B']:
#             print("Invalid choice. Defaulting to A.")
#             choice = 'A'
#         s1 = 0 if choice == 'A' else 1
#         result = task.step(s1)
#         state_name = '2A' if result['stage2_state'] == 0 else '2B'
#         print(f"  -> {result['transition'].upper()} transition to State {state_name}")
#         print(f"  -> Reward: {'YES (+1)' if result['reward'] else 'no (0)'}")
#         total_reward += result['reward']
#         print(f"  Total so far: {total_reward}/{t+1}")
#         print()
#     
#     print(f"\nFinal score: {total_reward}/{n_trials} ({100*total_reward/n_trials:.0f}%)")
#     return history
#
# # play_two_step(20)

print("Exercise 3: Uncomment the code above to play the task interactively.")
print("Reflect on whether your own strategy is model-free or model-based!")

### Exercise 4 (Advanced): Reward Rates Over Time

Compare the three agents (MF, MB, AIF) on reward rate over time using a sliding window. This shows whether the model-based advantage is consistent or emerges only in certain phases.

In [ ]:
# ── Exercise 4: Reward Rate Comparison ─────────────────────────────────────────
mf_rew = np.array([h['reward'] for h in mf_history])
mb_rew = np.array([h['reward'] for h in mb_history])
aif_rew = np.array([h['reward'] for h in aif_history])

window = 50
mf_smooth = np.convolve(mf_rew, np.ones(window)/window, mode='valid')
mb_smooth = np.convolve(mb_rew, np.ones(window)/window, mode='valid')
aif_smooth = np.convolve(aif_rew, np.ones(window)/window, mode='valid')

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(mf_smooth, color=COLORS['rl'], linewidth=2, label=f'Model-Free (mean={mf_rew.mean():.3f})', alpha=0.85)
ax.plot(mb_smooth, color=COLORS['reward'], linewidth=2, label=f'Model-Based (mean={mb_rew.mean():.3f})', alpha=0.85)
ax.plot(aif_smooth, color=COLORS['aif'], linewidth=2, label=f'Active Inference (mean={aif_rew.mean():.3f})', alpha=0.85)
ax.axhline(0.5, color='gray', linestyle=':', alpha=0.4, label='Chance')
ax.set_xlabel('Trial')
ax.set_ylabel('Reward Rate (50-trial window)')
ax.set_title('Reward Rate Over Time: Three Agent Types', fontsize=13)
ax.legend(loc='lower right', fontsize=9)
ax.set_ylim(0.3, 0.85)
plt.tight_layout()
plt.show()

## A2.9 Summary

### Key Takeaways

| Feature | Model-Free | Model-Based | Active Inference |
|---|---|---|---|
| **Internal model** | No (cached Q-values) | Yes (transition matrix) | Yes (B-matrix, required) |
| **Stage 1 computation** | Look up $Q_1(a)$ | $\sum_s T(s|a) \max Q_2(s,a')$ | $\sum_s B(s|a) \cdot C$-alignment |
| **After rare + reward** | Stay (reinforced) | Switch (other action leads there commonly) | Switch (B-matrix encodes transitions) |
| **Stay-probability pattern** | Parallel lines | Crossing lines | Crossing lines |
| **Interaction effect** | Near zero | Large positive | Large positive |
| **Cognitive cost** | Low (habit) | High (planning) | High (inference) |
| **Psychiatric relevance** | Compulsivity, addiction | Healthy goal-directed control | Unified framework |

### The Deep Insight

The Daw two-step task is not just a clever experiment — it is a **litmus test** for whether an agent has an internal model of the world. Active Inference passes this test by construction: you cannot compute expected free energy without a generative model. This is why the Free Energy Principle and Active Inference are often described as formalizations of Tolman's cognitive map hypothesis.

The hybrid parameter $w$ provides a continuous measure of where an agent (or a person) falls on the MF-MB spectrum, making it a powerful tool for computational psychiatry.

---

## Further Reading

1. **Daw, N.D., Gershman, S.J., Seymour, B., Dayan, P., & Dolan, R.J. (2011).** *Model-based influences on humans' choices and striatal prediction errors.* Neuron, 69(6), 1204-1215. The original paper introducing the two-step task. Essential reading — the experimental design is a masterclass in computational cognitive neuroscience.

2. **Dolan, R.J. & Dayan, P. (2013).** *Goals and habits in the brain.* Neuron, 80(2), 312-325. A comprehensive review of the neural circuits supporting model-free and model-based control, including the role of prefrontal cortex, striatum, and dopamine.

3. **Gillan, C.M., Kosinski, M., Whelan, R., Phelps, E.A., & Daw, N.D. (2016).** *Characterizing a psychiatric symptom dimension related to deficits in goal-directed control.* eLife, 5, e11305. Shows that reduced model-based control (lower $w$) correlates with compulsivity across multiple psychiatric disorders.

4. **Kool, W., Cushman, F.A., & Gershman, S.J. (2016).** *When does model-based control pay off?* PLOS Computational Biology, 12(8). Analyzes the cost-benefit tradeoff of model-based planning: it is computationally expensive and only advantageous in environments with sufficient structure.

5. **Friston, K.J., FitzGerald, T., Rigoli, F., Schwartenbeck, P., & Pezzulo, G. (2017).** *Active Inference: A Process Theory.* Neural Computation, 29(1), 1-49. The foundational Active Inference paper. Section 4 discusses how AIF handles sequential decisions and subsumes model-based planning under variational inference.

---

**Navigation:**
- Back to [Module 5: Model-Based RL](05_model_based_rl.ipynb)
- Forward to [Module 13: The Rosetta Stone](13_rosetta_stone.ipynb)
- See also [Appendix A0: Supplementary References](A0_supplementary_references.ipynb)